# 05 — Inference and Explainability

CAGEFusion's co-attention architecture makes predictions interpretable.
This notebook shows how to use `GradientExplainer` to understand which
SMILES tokens and molecular descriptors drive a prediction, and how to
visualise the cross-attention weights.

**What you'll learn**
- Load and run the pipeline (recap)
- `GradientExplainer.explain` — token + auxiliary feature saliency
- Visualise token saliency as a bar chart
- Auxiliary feature saliency (which RDKit descriptors matter)
- Attention map plots via `plot_all_attention=True`
- Compare saliency across structurally related molecules

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cage_fusion import CageFusionPipeline
from cage_fusion.inference.explainer import GradientExplainer

## 1. Load the pipeline

In [ ]:
# Replace with a local checkpoint dir if you have one, e.g. "/tmp/cage_fusion_custom"
pipe = CageFusionPipeline.from_pretrained("sidxz/cage-fusion-nuisance")
print("Tasks:", pipe.tasks)

## 2. Single SMILES prediction — all output keys

In [ ]:
smiles = "CN(C)c1ccc(cc1)C(=C2C=CC(=[N+](C)C)C=C2)c3ccccc3"  # malachite green — known PAINS

result = pipe(smiles)
for key, value in result.items():
    print(f"{key:30s}: {value}")

## 3. Gradient saliency for one SMILES

In [ ]:
explainer = GradientExplainer(pipe)

# Choose the task you want to explain
target_task = pipe.tasks[0]

explanation = explainer.explain(smiles, target_task=target_task)

print(f"SMILES      : {explanation['smiles']}")
print(f"Task        : {explanation['task']}")
print(f"Probability : {explanation['probability']:.4f}")
print(f"Prediction  : {explanation['predicted_class']} (threshold={explanation['threshold']:.3f})")
print(f"\nTop tokens:")
tok_sal = sorted(zip(explanation['tokens'], explanation['token_saliency']),
                 key=lambda x: -x[1])
for tok, sal in tok_sal[:10]:
    print(f"  {tok:15s}  {sal:.4f}")

## 4. Token saliency bar chart

In [ ]:
tokens = explanation["tokens"]
saliency = np.array(explanation["token_saliency"])

# Top 15 tokens by saliency
top_idx = np.argsort(saliency)[::-1][:15]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh([tokens[i] for i in top_idx[::-1]], saliency[top_idx[::-1]], color="steelblue")
ax.set_xlabel("Gradient saliency")
ax.set_title(f"Token saliency for '{target_task}'")
plt.tight_layout()
plt.show()

## 5. Auxiliary feature saliency

Which RDKit physicochemical descriptors most influence the prediction?

In [ ]:
aux_sal = np.array(explanation["aux_saliency"])

# Descriptor names (first 217 RDKit descriptors, ordered as computed)
try:
    from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator
    from rdkit.Chem import Descriptors
    desc_names = [d[0] for d in Descriptors.descList[:len(aux_sal)]]
except Exception:
    desc_names = [f"feat_{i}" for i in range(len(aux_sal))]

top_aux_idx = np.argsort(aux_sal)[::-1][:15]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh([desc_names[i] for i in top_aux_idx[::-1]], aux_sal[top_aux_idx[::-1]], color="darkorange")
ax.set_xlabel("Gradient saliency")
ax.set_title(f"Auxiliary feature saliency for '{target_task}'")
plt.tight_layout()
plt.show()

## 6. Attention map visualisation

Pass `plot_all_attention=True` to generate PNG attention maps for every
molecule in the batch.  Images are written to `attn_plot_dir`.

In [ ]:
import os
from IPython.display import Image, display

attn_dir = "/tmp/attn_plots"

df_in = pd.DataFrame({"SMILES": [smiles]})

out_df = pipe.predict(
    df_in,
    plot_all_attention=True,
    attn_plot_dir=attn_dir,
)

# Show atom contribution map
atom_img = os.path.join(attn_dir, "idx_0", "atom_total_contrib.png")
if os.path.exists(atom_img):
    print("Atom total contribution:")
    display(Image(atom_img))

# Show functional-group attention map
fg_img = os.path.join(attn_dir, "idx_0", "fg_prompt_attention.png")
if os.path.exists(fg_img):
    print("Functional-group attention:")
    display(Image(fg_img))

## 7. Compare saliency across related molecules

In [ ]:
molecules = {
    "malachite green (PAINS)": "CN(C)c1ccc(cc1)C(=C2C=CC(=[N+](C)C)C=C2)c3ccccc3",
    "crystal violet (PAINS)" : "CN(C)c1ccc(cc1)C(c2ccc(cc2)N(C)C)(c3ccc(cc3)N(C)C)=C",
    "aspirin (clean)"         : "CC(=O)Oc1ccccc1C(=O)O",
}

rows = []
for name, smi in molecules.items():
    exp = explainer.explain(smi, target_task=target_task)
    top_toks = sorted(zip(exp["tokens"], exp["token_saliency"]), key=lambda x: -x[1])[:3]
    rows.append({
        "molecule": name,
        "probability": round(exp["probability"], 3),
        "predicted_class": exp["predicted_class"],
        "top_3_tokens": ", ".join(t for t, _ in top_toks),
    })

pd.DataFrame(rows)

## 8. Batch DataFrame prediction with base64 attention images

In [ ]:
import base64
from IPython.display import HTML

df_batch = pd.DataFrame({"SMILES": list(molecules.values()),
                          "name": list(molecules.keys())})

out_batch = pipe.predict(
    df_batch,
    plot_all_attention=True,
    attn_plot_dir="/tmp/attn_batch",
)

# Display embedded images inline
html_parts = []
for i, row in out_batch.iterrows():
    b64 = row.get("atom_total_contrib_base64", "")
    if b64:
        html_parts.append(
            f'<div style="display:inline-block; margin:10px">'
            f'<p>{row["SMILES"][:30]}…</p>'
            f'<img src="data:image/png;base64,{b64}" width="300"/>'
            f'</div>'
        )

HTML("".join(html_parts)) if html_parts else print("No images generated.")